# Canonical Model 05: Modern PEST Setup

This notebook shows the **declarative calibration facade** added to `PestProject`: a few readable lines — `parameterize`, `observe`, `forecast`, `build` — that compile straight to native `pyemu.utils.PstFrom`. No hand-rolled template files, and pyEMU's own `apply_list_and_array_pars` drives the forward run.

We calibrate the **canonical valley model itself**. Its hydraulic conductivity is deliberately reset to a wrong starting value (3x too transmissive), and the head observations are sampled from the true K field, so there is real work for calibration to do.

In [1]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo
from myflopy.modflow.mf6.pest import PestProject

notebook_header(
    '05',
    'Modern PEST Setup',
    'Declare parameters, observations, and forecasts, then build a native PEST++ control file.',
)

## 1. Build the canonical calibration demo

`build_canonical_calibration_demo` builds the canonical valley model as the synthetic **truth**, samples head observations from many wells across the valley floor (plus one down-valley head forecast near the lake), then resets K to a wrong, uniformly-too-high starting value. The returned `model` is the **starting** model; the targets carry the truth-derived measured heads.

In [2]:
artifact_root = Path('../artifacts/canonical_pest_modern')
artifact_root.mkdir(parents=True, exist_ok=True)

demo = build_canonical_calibration_demo(artifact_root / 'setup_model')

pd.Series({
    'cells': int(demo.model.vor.ncpl),
    'head_observations': len(demo.head_targets.to_long()),
    'observation_wells': demo.head_targets.locations_gdf.shape[0],
    'start_K_factor': demo.start_k_factor,
    'forecast_points': demo.forecast_targets.locations_gdf.shape[0],
}, name='calibration demo')

VoronoiGrid initializing.
Voronoi grid initialized.


getting connectivity properties (iac, ja, cl12, hwva, nja)


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model viz_prt_master...
    writing model name file...
    writing package disv...


    writing package ic...
    writing package npf...
    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 200 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 200 based on size of stress_period_data
    writing package rch...


...

    Solving:  Stress period:     5    Time step:     2


    Solving:  Stress period:     6    Time step:     1


    Solving:  Stress period:     6    Time step:     2


 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/22 21:59:59
 Elapsed run time:  6.058 Seconds
 
 Normal termination of simulation.

Success is:  True


cells                2500.0
head_observations      96.0
observation_wells      16.0
start_K_factor          3.0
forecast_points         1.0
Name: calibration demo, dtype: float64

## 2. Declare parameters

`parameterize(target, ...)` records what to adjust. `bounds` are the multiplier range; `physical` clamps the *final* model value after multiplying (so calibration can't push K somewhere nonphysical). Here we adjust a single constant K multiplier and a constant recharge multiplier. The canonical model is multi-layer, so one constant K multiplier scales every layer's K file together.

> Spatial pilot points for array K need a Voronoi cell spatial reference and are coming in a later phase; `constant` and `zone` work today.

In [3]:
cal = PestProject(
    model=demo.model,
    name='canonical_demo',
    workspace=artifact_root / 'setup_template',
    start_datetime='2024-01-01',
)

cal.parameterize('k',        style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0))
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0),  physical=(0.0, 1e-2))

NativeParameterSpec(target='recharge', style='constant', bounds=(0.3, 3.0), physical=(0.0, 0.01), transform='log', additive=False, zones=None, layers=None, pp_space=None, pp_points=None, correlation=None, temporal=None, name='recharge', capture=False, extra={}, resolved_files=[])

## 3. Observations and forecasts

`observe` registers history-matching targets; `forecast` registers predictions of interest. Both take the **same** myflopy target objects — a forecast is simply an observation you predict but never match, so it is set to zero weight and recorded as a PEST++ forecast for later uncertainty analysis.

In [4]:
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)

HeadTargetObservationSpec(targets=HeadTargets(locations=[{'name': 'fore_lakehead', 'layer': 0, 'cell': 1231}], values=   per fore_lakehead
0    0     94.257655
1    1     94.962035
2    2     95.402811
3    3     95.577649
4    4     95.642142
5    5     95.595344, name_column='name', layer_column='layer', group_column='group', weight_column='weight', time_column='per', value_column='head', times=None), simulated_values=None, prefix='fore1')

## 4. Review the resolved configuration (before building)

`cal.settings()` is the anti-black-box: it prints exactly what was declared so you can sanity-check before committing to a build.

In [5]:
print(cal.settings())

PEST calibration: canonical_demo  (model: viz_prt_master)
  template : ..\artifacts\canonical_pest_modern\setup_template
  start    : 2024-01-01
  parameters (2):
    - k          style=constant    bounds=(0.05, 2.0) physical=(0.01, 300.0) transform=log
    - recharge   style=constant    bounds=(0.3, 3.0) physical=(0.0, 0.01) transform=log
  observations (1):
    - hds        kind=headtarget   n=None
  forecasts (1):
    - fore1      n=None
  (not built yet -- call cal.build())


## 5. Build the control file

`build` externalizes the model inputs, registers parameters natively, wires the forward run, and writes the `.pst`. With `noptmax=0` the run is configured to evaluate the model once and compute residuals — the standard sanity check. `settings()` now also reports the control-file counts.

In [6]:
pst = cal.build('canonical_demo.pst', noptmax=0)
print(cal.settings())

PEST calibration: canonical_demo  (model: viz_prt_master)
  template : ..\artifacts\canonical_pest_modern\setup_template
  start    : 2024-01-01
  parameters (2):
    - k          style=constant    bounds=(0.05, 2.0) physical=(0.01, 300.0) transform=log
    - recharge   style=constant    bounds=(0.3, 3.0) physical=(0.0, 0.01) transform=log
  observations (1):
    - hds        kind=headtarget   n=96
  forecasts (1):
    - fore1      n=6
  built control file:
    npar=2 (groups=2)  nobs=102 (nonzero weight=96)  forecasts=6  noptmax=0


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\pyemu\utils\pst_from.py:1281: PyemuWarning: add_py_function(): _write_head_target_csv already in forward run python functions, not overriding here, original will be maintained


## 6. Inspect the generated PEST control problem

Everything below is read from the actual `.pst` pyEMU wrote — the parameter groups, bounds, and observation weights PEST++ will use.

In [7]:
display(cal.settings().parameter_frame())
display(
    pst.parameter_data[['parnme', 'pargp', 'partrans', 'parval1', 'parlbnd', 'parubnd']]
    .groupby('pargp').agg(
        parameters=('parnme', 'count'),
        transform=('partrans', 'first'),
        lower_bound=('parlbnd', 'min'),
        upper_bound=('parubnd', 'max'),
    )
)

,target,style,bounds,physical,transform,additive,name
0,k,constant,"(0.05, 2.0)","(0.01, 300.0)",log,False,k
1,recharge,constant,"(0.3, 3.0)","(0.0, 0.01)",log,False,recharge


,parameters,transform,lower_bound,upper_bound
pargp,,,,
k,1,log,0.05,2.0
recharge,1,log,0.30,3.0


## 7. Validate the forward run

Every PEST iteration calls `forward_run.py`. Running it once directly is the most important setup check: it must apply the parameter multipliers, run MODFLOW, and regenerate the simulated-observation files. Note that `apply_list_and_array_pars` — pyEMU's native apply — is present, not stripped.

In [8]:
import subprocess, sys

template = cal.template_workspace
result = subprocess.run([sys.executable, 'forward_run.py'], cwd=template,
                        capture_output=True, text=True)
assert result.returncode == 0, result.stdout + '\n' + result.stderr

forward_run_text = (template / 'forward_run.py').read_text()
pd.Series({
    'forward_run_returncode': result.returncode,
    'apply_list_and_array_pars_present': 'apply_list_and_array_pars' in forward_run_text,
    'mult2model_info_written': (template / 'mult2model_info.csv').exists(),
    'simulated_heads_regenerated': (template / 'hds_simulated_heads.csv').exists(),
}, name='forward-run validation')

forward_run_returncode                  0
apply_list_and_array_pars_present    True
mult2model_info_written              True
simulated_heads_regenerated          True
Name: forward-run validation, dtype: object

## What's next

- The control file, `forward_run.py`, templates, and multiplier files now form a self-contained PEST++ setup.
- Drop to the raw pyEMU objects any time with `cal.pf` and `cal.pst`.
- **Notebook 06** runs PESTPP-IES on this exact setup with one line (`cal.run_ies(...)`) and assesses the result — phi convergence, ensembles vs observations, and posterior forecast uncertainty.